In [38]:
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, RobustScaler, OrdinalEncoder, LabelEncoder
)
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_squared_error
from matplotlib import pyplot as plt

from sklearn.model_selection import (
    cross_validate, GridSearchCV, ShuffleSplit
)

In [3]:
# leitura do conjunto de dados
df = sns.load_dataset('tips')

# informações sobre as variáveis
nominal_columns = ['sex', 'smoker', 'day', 'time']
continuos_columns = ['total_bill', 'size']
target_column = 'tip'

In [5]:
X = df.drop(columns=[target_column], axis=1)
y = df[target_column]

In [10]:
nominal_preprocessor = Pipeline(steps=[
    # Tratamento de dados faltantes ()
    ('missing', SimpleImputer(strategy='most_frequent')),
    # Codificação de variáveis
    ('encoding', OneHotEncoder(sparse=False, handle_unknown='ignore')), 
    # Normalização de variáveis
     ('normalization', StandardScaler()) 
])
continuous_preprocessor = Pipeline(steps=[
    # Tratamento de dados faltantes
    ('missing', KNNImputer(n_neighbors=5)),
    # Normalização
    ('normalization', RobustScaler()) 
])

preprocessor = ColumnTransformer([
    ('nominal', nominal_preprocessor, nominal_columns),
    ('continuos', continuous_preprocessor, continuos_columns)
])

In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=.2,
    # random_state=42
)

In [67]:
# validação cruzada hold-out
linear_model = Ridge()
svr_model = SVR()

# 1. separar conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=.2,
    random_state=42
)

# preprocessamento de dados
preprocessor.fit(X_train)
X_train = preprocessor.transform(X_train)
X_test = preprocessor.transform(X_test)

# 2. treinar usando o conjunto de treino
#    e testar usando o conjunto de teste

linear_model.fit(X_train, y_train)
svr_model.fit(X_train, y_train)

# 3. calcular métrica comparando as predições com os valores reais

mse_linear = mean_squared_error(
    y_test,
    linear_model.predict(X_test)
)
mse_svr = mean_squared_error(
    y_test,
    svr_model.predict(X_test)
)

print(f"MSE Linear {mse_linear:.4f}")
print(f"MSE SVR:   {mse_svr:.4f}")

MSE Linear 0.7022
MSE SVR:   0.7902


In [102]:
# validação cruzada k-fold

# 1. separar conjuntos de treino e teste
kfold = KFold(
    n_splits=100,
    shuffle=True,
    random_state=42
)

mse_linear_list = []
mse_svr_list = []
for k, (train_index, test_index) in enumerate(kfold.split(X)):
    X_train = X.iloc[train_index, :]
    y_train = y.iloc[train_index]
    X_test = X.iloc[test_index, :]
    y_test = y.iloc[test_index]
    
    preprocessor.fit(X_train)
    X_train = preprocessor.transform(X_train)
    X_test = preprocessor.transform(X_test)

    linear_model.fit(X_train, y_train)
    svr_model.fit(X_train, y_train)
    
    mse_linear = mean_squared_error(y_test, linear_model.predict(X_test))
    mse_svr = mean_squared_error(y_test, svr_model.predict(X_test))
    
    mse_linear_list.append(mse_linear)
    mse_svr_list.append(mse_svr)
    
print("Linear")
print(f" - AVG:  {np.mean(mse_linear_list):.4f}")
print(f" - STD:  {np.std(mse_linear_list):.4f}")
print("SVR:   ")
print(f" - AVG:  {np.mean(mse_svr_list):.4f}")
print(f" - STD:  {np.std(mse_svr_list):.4f}")

Linear
 - AVG:  1.0767
 - STD:  1.3209
SVR:   
 - AVG:  1.1113
 - STD:  1.8280


In [106]:
models = {
    'linear': Ridge(),
    'svr': SVR(),
}

results = {}
for model_name, model in models.items():
    kfold = KFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )
    approach = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    metrics = cross_validate(
        approach, 
        X, 
        y, 
        cv=kfold, 
        scoring=['neg_mean_squared_error', 'r2']
    )
    metrics
    metrics['model'] = np.array([model_name] * len(metrics['fit_time']))
    if not results:
        results = metrics
    else:
        for key in results.keys():
            results[key] = results[key].tolist()
            results[key].extend(metrics[key])

In [110]:
pd.DataFrame(results).groupby("model").agg(["mean", "std"]).transpose()

model                               linear       svr
fit_time                    mean  0.038608  0.040553
                            std   0.021558  0.015630
score_time                  mean  0.018859  0.014503
                            std   0.010883  0.002650
test_neg_mean_squared_error mean -1.101493 -1.152142
                            std   0.538629  0.697756
test_r2                     mean  0.399315  0.382836
                            std   0.208219  0.215192

In [111]:
# conjunto de modelos e seus hiperparâmetros
models = [
    (
        "DT",
        DecisionTreeRegressor(),
        {
            "max_depth": [1, 5, 9, 12],
            "min_samples_leaf": [1, 5, 9],
            "min_weight_fraction_leaf": np.linspace(0.1, 0.9, 2),
            "max_features": ["auto","log2"],
            "max_leaf_nodes": [None,10]
        }

    ),
    (
        "RR",
        Ridge(),
        {
            "alpha": np.linspace(0, 1, 2)
        }
    ),
    (
        "SVR",
        SVR(max_iter=10000),
        {
            "C": np.geomspace(0.1, 100, 2),
            "gamma": np.geomspace(1, 0.001, 2),
            "kernel": ["rbf", "poly", "sigmoid"]
        }
    ),
    (
        "KNN",
        KNeighborsRegressor(),
        {
            "n_neighbors": np.arange(2, 10, 3),
            "weights": ["uniform","distance"]
        }
    )
]

In [112]:
n_splits = 10
final_results = {}
for model_name, model, model_hparams_grid in models:
    print(f"{model_name} is running...")
    model_gs = GridSearchCV(
        model,
        model_hparams_grid,
        scoring='neg_root_mean_squared_error',
        cv=5
    )
    approach = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model_gs)
    ])
    results = cross_validate(
        approach,
        X=X,
        y=y,
        # https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter
        scoring=[
            "neg_root_mean_squared_error",
            "neg_mean_absolute_error",
            "neg_median_absolute_error",
            "r2"
        ],
        cv=ShuffleSplit(n_splits=n_splits, test_size=.2)
    )
    results["name"] = [model_name] * n_splits
    if final_results:
        for key, value in results.items():
            final_results[key] = np.append(final_results[key], value)
    else:
        final_results = results

DT is running...
RR is running...
SVR is running...
KNN is running...


In [113]:
(
    pd
    .DataFrame(final_results)
    .groupby('name')
    .agg(['mean', 'std'])
    .transpose()
)

name                                         DT       KNN        RR       SVR
fit_time                         mean  0.923703  0.159927  0.076252  0.806328
                                 std   0.099008  0.010053  0.015963  0.058999
score_time                       mean  0.017004  0.015722  0.015689  0.017295
                                 std   0.005064  0.002291  0.003695  0.005053
test_neg_root_mean_squared_error mean -1.188348 -1.122975 -0.998322 -1.093478
                                 std   0.219472  0.237369  0.154793  0.117757
test_neg_mean_absolute_error     mean -0.888571 -0.810585 -0.734847 -0.778332
                                 std   0.133487  0.116733  0.086810  0.075415
test_neg_median_absolute_error   mean -0.682978 -0.577220 -0.538716 -0.527099
                                 std   0.194635  0.095478  0.099338  0.111095
test_r2                          mean  0.213791  0.226309  0.459809  0.376602
                                 std   0.168212  0.169815  0.127456  0.259076

In [ ]:
# # conjunto de modelos e seus hiperparâmetros
# models = [
#     (
#         "DT",
#         DecisionTreeRegressor(),
#         {
#             "splitter": ["best","random"],
#             "max_depth": [1,3,5,7,9,11,12],
#             "min_samples_leaf": [1,2,3,4,5,6,7,8,9,10],
#             "min_weight_fraction_leaf": [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
#             "max_features": ["auto","log2","sqrt",None],
#             "max_leaf_nodes": [None,10,20,30,40,50,60,70,80,90]
#         }

#     ),
#     (
#         "RR",
#         Ridge(),
#         {
#             "alpha": np.arange(0, 1, 0.01)
#         }
#     ),
#     (
#         "SVR",
#         SVR(),
#         {
#             "C": [0.1,1, 10, 100],
#             "gamma": [1,0.1,0.01,0.001],
#             "kernel": ["rbf", "poly", "sigmoid"]
#         }
#     ),
#     (
#         "KNN",
#         KNeighborsRegressor(),
#         {
#             "n_neighbors": [2,3,4,5,6],
#             "weights": ["uniform","distance"]
#         }
#     )
# ]